# Notebook 08: Seed Ensemble (Model Averaging)

## Motivation

In Notebook 03, the multi-seed robustness test revealed a **CV of 8.90%** (borderline PASS at threshold < 10%), with Seed 123 producing an outlier RMSE of 7.61 vs the mean of 6.59. The champion model (Seed 42) was deployed as a single point estimate.

**Model averaging** (ensembling) reduces the sensitivity to random initialisation by combining predictions from multiple independently trained models. This is equivalent to **credibility pooling across models** — each model sees the same data but learns slightly different representations due to different weight initialisations and dropout masks during training.

## Methodology

1. **Retrain** the champion architecture (LSTM 48-32, lb=15, lr=0.001, λ_coh=0.001, λ_mono=0.001) with 5 different seeds: [42, 123, 256, 512, 1024].
2. **Save** all 5 models.
3. **For each model**, run MC Dropout forecast (1,000 simulations × 30 years).
4. **Average** the scaled predictions across the 5 models before inverse transform and noise addition.
5. **Apply corrected σ** (from Notebook 07, walk-forward residuals) as process noise.
6. **Reconstruct** e₀, compute CI, SCR, and compare with single-seed results.

## Why Average in Scaled Space

All 5 models share the same scaler (fitted on the same training data with the same seed for data preparation). Averaging their scaled predictions preserves the neural signal while cancelling out seed-specific noise. The inverse transform and process noise are applied once to the ensemble mean.

## Computational Budget

- Training: ~10 min × 5 seeds = ~50 min
- MC Dropout: ~3 min × 5 seeds × 2 sexes = ~30 min
- Total: ~80 min (~1.3 hours on M1 Pro)

We use 200 simulations per model (not 1,000) because the ensemble of 5 models × 200 simulations = 1,000 total trajectories — matching the sample size of NB04/07. With 5 independent models, 200 simulations per model are sufficient to capture each model's epistemic uncertainty profile; the cross-model averaging provides the additional stabilisation.

This is a one-time investment. The resulting ensemble assets are saved for all downstream use.


## 8.1: Setup & Configuration

In [ ]:
import sys
sys.path.append('../src')
from reproducibility import set_seed, get_seed_list

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os, logging, warnings, time
warnings.filterwarnings('ignore')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel(logging.ERROR)
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
import joblib

from style_config import set_style, save_dual, COUNTRIES, COUNTRY_COLORS
set_style("notebook")

PROCESSED_DIR = "../data/processed/"
MODELS_DIR = "../models/"
FIGURES_DIR = "../reports/figures/"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Load Li-Lee parameters
with open(os.path.join(PROCESSED_DIR, "li_lee_params.pkl"), "rb") as f:
    bundle = pickle.load(f)

feature_matrices = bundle["feature_matrices"]
common_factors = bundle["common_factors"]
YEARS = bundle["metadata"]["years"]
AGES = bundle["metadata"]["ages"]
N_AGES = len(AGES)

# Load training metadata (champion config from NB03)
with open(os.path.join(PROCESSED_DIR, "training_meta.pkl"), "rb") as f:
    meta = pickle.load(f)

# Load corrected σ from NB07
with open(os.path.join(PROCESSED_DIR, "forecasting_assets_corrected.pkl"), "rb") as f:
    nb07_assets = pickle.load(f)

sigma_res_male = nb07_assets['process_std_male']
sigma_res_female = nb07_assets['process_std_female']

# Champion configuration
LOOKBACK = meta['lookback']
UNITS_L1 = meta['units_l1']
UNITS_L2 = meta['units_l2']
LR = meta['lr']
LAMBDA_COH = meta['lambda_coherence']
LAMBDA_MONO = meta['lambda_monotonicity']
BATCH_SIZE = 8
EPOCHS = 150
TRAIN_SPLIT_IDX = meta['train_split_idx']
N_FEATURES = meta['n_features']

SEEDS = get_seed_list(5)
N_SIMULATIONS = 200
N_YEARS_AHEAD = 30

print(f"Champion config: LSTM({UNITS_L1},{UNITS_L2}), lb={LOOKBACK}, lr={LR}")
print(f"  λ_coh={LAMBDA_COH}, λ_mono={LAMBDA_MONO}")
print(f"Seeds: {SEEDS}")
print(f"Corrected σ (Male Kt): {sigma_res_male[0]:.4f}")
print(f"Corrected σ (Female Kt): {sigma_res_female[0]:.4f}")

## 8.2: Data Preparation & Loss Function

Reproduced exactly from NB03 to ensure identical training conditions.

In [ ]:
def prepare_joint_data(lookback):
    """Prepare joint M/F sequences — identical to NB03."""
    male_data = np.column_stack([feature_matrices['male'], np.zeros(len(feature_matrices['male']))])
    female_data = np.column_stack([feature_matrices['female'], np.ones(len(feature_matrices['female']))])
    train_combined = np.vstack([male_data[:TRAIN_SPLIT_IDX], female_data[:TRAIN_SPLIT_IDX]])
    scaler = StandardScaler()
    scaler.fit(train_combined)
    male_scaled = scaler.transform(male_data)
    female_scaled = scaler.transform(female_data)
    def make_seq(data, lb):
        X, y = [], []
        for i in range(lb, len(data)):
            X.append(data[i-lb:i])
            y.append(data[i])
        return np.array(X), np.array(y)
    X_m, y_m = make_seq(male_scaled, lookback)
    X_f, y_f = make_seq(female_scaled, lookback)
    n_train = TRAIN_SPLIT_IDX - lookback
    X_train = np.concatenate([X_m[:n_train], X_f[:n_train]])
    y_train = np.concatenate([y_m[:n_train], y_f[:n_train]])
    X_val = np.concatenate([X_m[n_train:], X_f[n_train:]])
    y_val = np.concatenate([y_m[n_train:], y_f[n_train:]])
    idx = np.random.permutation(len(X_train))
    X_train, y_train = X_train[idx], y_train[idx]
    return X_train, y_train, X_val, y_val, scaler

class AINNLoss(keras.losses.Loss):
    """AINN loss: MSE + coherence + monotonicity — identical to NB03."""
    def __init__(self, lambda_coherence=0.0, lambda_monotonicity=0.0, **kwargs):
        super().__init__(**kwargs)
        self.lambda_coherence = lambda_coherence
        self.lambda_monotonicity = lambda_monotonicity
    def call(self, y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred))
        coherence = tf.reduce_mean(tf.square(y_pred[:, 1:7]))
        monotonicity = tf.reduce_mean(tf.square(tf.nn.relu(y_pred[:, 0])))
        return mse + self.lambda_coherence * coherence + self.lambda_monotonicity * monotonicity
    def get_config(self):
        config = super().get_config()
        config.update({"lambda_coherence": self.lambda_coherence,
                       "lambda_monotonicity": self.lambda_monotonicity})
        return config

print("Data preparation and loss function ready.")

## 8.3: Train 5 Models (One Per Seed)

Each model is trained from scratch with the champion configuration, varying only the random seed. All models are saved for reproducibility.

In [ ]:
models = {}
seed_rmses = {}

print("Training 5 models (one per seed)...")
print("=" * 60)

t_start_all = time.time()

for seed in SEEDS:
    t_start = time.time()
    print(f"\n--- Seed {seed} ---")
    
    # set_seed BEFORE data preparation (controls shuffle order)
    set_seed(seed)
    X_train, y_train, X_val, y_val, scaler_seed = prepare_joint_data(LOOKBACK)
    
    # Build model
    set_seed(seed)
    inputs = layers.Input(shape=(LOOKBACK, N_FEATURES))
    x = layers.LSTM(UNITS_L1, return_sequences=True)(inputs)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(UNITS_L2)(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(N_FEATURES)(x)
    model = keras.Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=AINNLoss(lambda_coherence=LAMBDA_COH, lambda_monotonicity=LAMBDA_MONO)
    )
    
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=0)
    history = model.fit(
        X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val), callbacks=[es], verbose=0
    )
    
    # Evaluate
    pred = scaler_seed.inverse_transform(model.predict(X_val, verbose=0))
    true = scaler_seed.inverse_transform(y_val)
    rmse = float(np.sqrt(np.mean((true - pred) ** 2)))
    
    elapsed = time.time() - t_start
    print(f"  RMSE: {rmse:.4f} | Epochs: {len(history.history['loss'])} | Time: {elapsed:.0f}s")
    
    # Save model
    model_path = os.path.join(MODELS_DIR, f"ainn_seed_{seed}.keras")
    model.save(model_path)
    
    models[seed] = model
    seed_rmses[seed] = rmse

total_time = time.time() - t_start_all

print(f"\n{'=' * 60}")
print(f"All 5 models trained in {total_time/60:.1f} minutes.")
print(f"\nRMSE by seed:")
for seed, rmse in seed_rmses.items():
    marker = " ← champion" if seed == 42 else ""
    print(f"  Seed {seed:4d}: {rmse:.4f}{marker}")

mean_rmse = np.mean(list(seed_rmses.values()))
std_rmse = np.std(list(seed_rmses.values()))
cv = std_rmse / mean_rmse * 100
print(f"\nMean: {mean_rmse:.4f}, Std: {std_rmse:.4f}, CV: {cv:.2f}%")

# Save the shared scaler (all seeds produce the same scaler due to identical data)
# Verify this:
set_seed(42)
_, _, _, _, scaler_ref = prepare_joint_data(LOOKBACK)
joblib.dump(scaler_ref, os.path.join(MODELS_DIR, "scaler_joint.pkl"))
print(f"\nModels saved to {MODELS_DIR}ainn_seed_*.keras")

## 8.4: Ensemble MC Dropout Forecast

For each of the 5 models, we run 1,000 MC Dropout simulations. The ensemble prediction at each simulation × step is the **mean across the 5 models' scaled predictions**.

This is equivalent to a Bayesian model combination where each model receives equal weight (uniform prior over initialisations). The averaging cancels out seed-specific artefacts while preserving the shared mortality signal.

**Note on simulation count**: We run 200 simulations per model. With 5 models, this yields 5 × 200 = 1,000 total trajectories — the same sample size as NB04/07. Each model contributes its own epistemic uncertainty profile via MC Dropout; the ensemble averaging across seeds adds model-selection robustness on top.

In [ ]:
# Load the reference scaler (same for all seeds)
scaler = joblib.load(os.path.join(MODELS_DIR, "scaler_joint.pkl"))

def get_initial_sequence(sex, lookback=LOOKBACK):
    sex_indicator = 0.0 if sex == 'male' else 1.0
    data = np.column_stack([
        feature_matrices[sex],
        np.full(len(feature_matrices[sex]), sex_indicator)
    ])
    data_scaled = scaler.transform(data)
    return data_scaled[-lookback:]

def forecast_mcd_single(model, initial_sequence, n_steps=N_YEARS_AHEAD, n_sims=N_SIMULATIONS):
    """MC Dropout forecast for a single model. Returns scaled predictions."""
    n_features = initial_sequence.shape[-1]
    all_sims = np.zeros((n_sims, n_steps, n_features))
    for sim in range(n_sims):
        current_seq = initial_sequence.copy()
        for step in range(n_steps):
            inp = tf.convert_to_tensor(current_seq[np.newaxis, ...], dtype=tf.float32)
            pred = model(inp, training=True).numpy().reshape(n_features)
            all_sims[sim, step, :] = pred
            current_seq = np.roll(current_seq, -1, axis=0)
            current_seq[-1] = pred
        if (sim + 1) % 100 == 0:
            print(f"      {sim+1}/{n_sims} done")
    return all_sims

init_male = get_initial_sequence('male')
init_female = get_initial_sequence('female')

# Run forecasts for all 5 models
all_forecasts_male = []   # list of (1000, 30, 8) arrays, one per seed
all_forecasts_female = []

t_start = time.time()

for seed in SEEDS:
    print(f"\n--- Seed {seed}: MC Dropout forecast ---")
    model = models[seed]
    
    print(f"  Male...")
    sims_m = forecast_mcd_single(model, init_male)
    all_forecasts_male.append(sims_m)
    
    print(f"  Female...")
    sims_f = forecast_mcd_single(model, init_female)
    all_forecasts_female.append(sims_f)

elapsed = time.time() - t_start
print(f"\nAll forecasts complete in {elapsed/60:.1f} minutes.")
print(f"Each array: {all_forecasts_male[0].shape} (sims, steps, features)")

## 8.5: Ensemble Averaging & Noise Addition

We average the 5 models' predictions **in scaled space** (simulation-wise), then inverse transform and add the corrected process noise from NB07.

In [ ]:
# Stack and average: (5, 1000, 30, 8) → mean over axis 0 → (1000, 30, 8)
ensemble_male_scaled = np.mean(np.stack(all_forecasts_male, axis=0), axis=0)
ensemble_female_scaled = np.mean(np.stack(all_forecasts_female, axis=0), axis=0)

print(f"Ensemble predictions: {ensemble_male_scaled.shape}")

# Inverse transform
def inverse_and_add_noise(sims_scaled, sigma_residual):
    n_sims, n_steps, n_features = sims_scaled.shape
    flat = sims_scaled.reshape(-1, n_features)
    flat_orig = scaler.inverse_transform(flat)
    sims_orig = flat_orig.reshape(n_sims, n_steps, n_features)
    sims_with_noise = sims_orig.copy()
    for t in range(n_steps):
        noise = np.random.normal(0, sigma_residual, size=(n_sims, n_features))
        sims_with_noise[:, t, :] += noise
    return sims_with_noise

set_seed(42)  # For reproducible noise
ens_male_orig = inverse_and_add_noise(ensemble_male_scaled, sigma_res_male)
ens_female_orig = inverse_and_add_noise(ensemble_female_scaled, sigma_res_female)

print(f"Ensemble with corrected noise: {ens_male_orig.shape}")

In [ ]:
# Level reconstruction & MBC (identical logic to NB04/07)

def integrate_to_levels(sims_diffs_orig, sex):
    sex_indicator = 0.0 if sex == 'male' else 1.0
    data = np.column_stack([
        feature_matrices[sex],
        np.full(len(feature_matrices[sex]), sex_indicator)
    ])
    last_level_2020 = data[-1]
    n_sims, n_steps, n_features = sims_diffs_orig.shape
    levels = np.zeros((n_sims, n_steps + 1, n_features))
    levels[:, 0, :] = last_level_2020
    for t in range(1, n_steps + 1):
        levels[:, t, :] = levels[:, t-1, :] + sims_diffs_orig[:, t-1, :]
    return levels

def apply_mbc(sims_diffs_orig, sex):
    data = np.column_stack([
        feature_matrices[sex],
        np.full(len(feature_matrices[sex]), 0.0 if sex == 'male' else 1.0)
    ])
    train_diffs = np.diff(data[:TRAIN_SPLIT_IDX], axis=0)
    mu_li_lee = train_diffs.mean(axis=0)
    mu_ainn = sims_diffs_orig.mean(axis=(0, 1))
    bias = mu_li_lee - mu_ainn
    sims_mbc = sims_diffs_orig + bias[np.newaxis, np.newaxis, :]
    return sims_mbc, bias

# Levels
ens_levels_male = integrate_to_levels(ens_male_orig, 'male')
ens_levels_female = integrate_to_levels(ens_female_orig, 'female')

# MBC
ens_male_mbc, ens_bias_male = apply_mbc(ens_male_orig, 'male')
ens_female_mbc, ens_bias_female = apply_mbc(ens_female_orig, 'female')
ens_levels_male_mbc = integrate_to_levels(ens_male_mbc, 'male')
ens_levels_female_mbc = integrate_to_levels(ens_female_mbc, 'female')

forecast_years = np.array([YEARS[-1]] + list(range(YEARS[-1]+1, YEARS[-1]+N_YEARS_AHEAD+1)))

print("Level reconstruction and MBC complete.")
print(f"Bias (Male Kt): {ens_bias_male[0]:.4f}")
print(f"Bias (Female Kt): {ens_bias_female[0]:.4f}")

## 8.6: Life Expectancy Reconstruction

In [ ]:
def calculate_e0(log_mx):
    mx = np.exp(log_mx)
    qx = 1.0 - np.exp(-mx)
    qx[-1] = 1.0
    px = 1.0 - qx
    lx = np.concatenate(([1.0], np.cumprod(px[:-1])))
    lx_extended = np.append(lx, 0.0)
    e0 = np.sum((lx_extended[:-1] + lx_extended[1:]) / 2.0)
    return e0

def reconstruct_e0_anchored(sims_diffs_orig, sex):
    Bx = common_factors[sex]['Bx']
    n_sims = sims_diffs_orig.shape[0]
    n_steps = sims_diffs_orig.shape[1]
    n_countries = len(COUNTRIES)
    delta_kt_all = sims_diffs_orig[:, :, 0]
    cumulative_delta_kt = np.cumsum(delta_kt_all, axis=1)
    zeros = np.zeros((n_sims, 1))
    cumulative_delta_kt = np.concatenate([zeros, cumulative_delta_kt], axis=1)
    e0_all = np.zeros((n_sims, n_steps + 1, n_countries))
    for c_idx, code in enumerate(COUNTRIES.keys()):
        log_mx_matrix = np.load(os.path.join(PROCESSED_DIR, f"{code}_log_mx_{sex}.npy"))
        log_mx_2020 = log_mx_matrix[:, -1]
        for s in range(n_sims):
            for t in range(n_steps + 1):
                log_mx_t = log_mx_2020 + Bx * cumulative_delta_kt[s, t]
                e0_all[s, t, c_idx] = calculate_e0(log_mx_t)
    return e0_all

print("Reconstructing e0 (ensemble, observation-anchored)...")
print("  Male, without MBC...")
ens_e0_male = reconstruct_e0_anchored(ens_male_orig, 'male')
print("  Male, with MBC...")
ens_e0_male_mbc = reconstruct_e0_anchored(ens_male_mbc, 'male')
print("  Female, without MBC...")
ens_e0_female = reconstruct_e0_anchored(ens_female_orig, 'female')
print("  Female, with MBC...")
ens_e0_female_mbc = reconstruct_e0_anchored(ens_female_mbc, 'female')

che_idx = list(COUNTRIES.keys()).index('CHE')
print(f"\ne0 arrays: {ens_e0_male.shape}")
print(f"Sanity check (CHE 2020): Male={ens_e0_male[:, 0, che_idx].mean():.2f}, Female={ens_e0_female[:, 0, che_idx].mean():.2f}")

## 8.7: Results — Single Seed (NB07) vs Ensemble

Compare three stages of improvement:
1. **Original NB04**: old σ, single seed
2. **NB07**: corrected σ, single seed
3. **NB08 (this)**: corrected σ, 5-seed ensemble

In [ ]:
# Load NB04 (old) and NB07 (corrected single-seed) for comparison
with open(os.path.join(PROCESSED_DIR, "forecasting_assets.pkl"), "rb") as f:
    nb04_assets = pickle.load(f)

comparison_rows = []
for c_idx, (code, name) in enumerate(COUNTRIES.items()):
    for sex, old_e0, nb07_e0, ens_e0 in [
        ('Male', nb04_assets['e0_male'], nb07_assets['e0_male'], ens_e0_male),
        ('Female', nb04_assets['e0_female'], nb07_assets['e0_female'], ens_e0_female)
    ]:
        e0_2020 = ens_e0[:, 0, c_idx].mean()
        
        # NB04 (original)
        old_p25 = np.percentile(old_e0[:, -1, c_idx], 2.5)
        old_p975 = np.percentile(old_e0[:, -1, c_idx], 97.5)
        old_ci = old_p975 - old_p25
        old_med = np.percentile(old_e0[:, -1, c_idx], 50)
        
        # NB07 (corrected σ, single seed)
        nb07_p25 = np.percentile(nb07_e0[:, -1, c_idx], 2.5)
        nb07_p975 = np.percentile(nb07_e0[:, -1, c_idx], 97.5)
        nb07_ci = nb07_p975 - nb07_p25
        nb07_med = np.percentile(nb07_e0[:, -1, c_idx], 50)
        
        # NB08 (ensemble)
        ens_p25 = np.percentile(ens_e0[:, -1, c_idx], 2.5)
        ens_p975 = np.percentile(ens_e0[:, -1, c_idx], 97.5)
        ens_ci = ens_p975 - ens_p25
        ens_med = np.percentile(ens_e0[:, -1, c_idx], 50)
        
        comparison_rows.append({
            'Country': name, 'Sex': sex,
            'e0(2020)': round(e0_2020, 2),
            'NB04 CI': round(old_ci, 2),
            'NB07 CI': round(nb07_ci, 2),
            'Ens CI': round(ens_ci, 2),
            'NB04 Med': round(old_med, 2),
            'NB07 Med': round(nb07_med, 2),
            'Ens Med': round(ens_med, 2),
            'Total CI Reduction': f"{(1 - ens_ci/old_ci)*100:.1f}%",
        })

df_comp = pd.DataFrame(comparison_rows)
print("=" * 120)
print("THREE-STAGE COMPARISON: NB04 (original) → NB07 (corrected σ) → NB08 (ensemble)")
print("=" * 120)
print(df_comp.to_string(index=False))

## 8.8: Updated SCR Estimates (Ensemble)

In [ ]:
scr_rows = []
for c_idx, (code, name) in enumerate(COUNTRIES.items()):
    for sex, e0_arr in [('Male', ens_e0_male), ('Female', ens_e0_female)]:
        e0_2050 = e0_arr[:, -1, c_idx]
        median_e0 = np.percentile(e0_2050, 50)
        var_995 = np.percentile(e0_2050, 99.5)
        scr_var = var_995 - median_e0
        threshold_99 = np.percentile(e0_2050, 99)
        tail_values = e0_2050[e0_2050 >= threshold_99]
        es_990 = tail_values.mean() if len(tail_values) > 0 else var_995
        scr_es = es_990 - median_e0
        scr_rows.append({
            'Country': name, 'Sex': sex,
            'Median e0 (2050)': round(median_e0, 2),
            'SCR VaR 99.5%': f'+{scr_var:.3f}',
            'SCR ES 99.0%': f'+{scr_es:.3f}',
        })

df_scr = pd.DataFrame(scr_rows)
print("=" * 80)
print("ENSEMBLE SCR ESTIMATES (Corrected σ + 5-Seed Ensemble)")
print("=" * 80)
print(df_scr.to_string(index=False))

print("\n--- SCR Progression (CHE) ---")
print("  NB04 (old σ, single seed):    Male +3.760, Female +2.919")
print("  NB07 (corrected σ, single):    Male +2.167, Female +1.606")
che_m = ens_e0_male[:, -1, che_idx]
che_f = ens_e0_female[:, -1, che_idx]
scr_m = np.mean(che_m[che_m >= np.percentile(che_m, 99)]) - np.percentile(che_m, 50)
scr_f = np.mean(che_f[che_f >= np.percentile(che_f, 99)]) - np.percentile(che_f, 50)
print(f"  NB08 (corrected σ, ensemble):  Male +{scr_m:.3f}, Female +{scr_f:.3f}")

## 8.9: Save Ensemble Assets

In [ ]:
ensemble_assets = {
    # Ensemble e0 trajectories
    "e0_male": ens_e0_male,
    "e0_female": ens_e0_female,
    "e0_male_mbc": ens_e0_male_mbc,
    "e0_female_mbc": ens_e0_female_mbc,
    # Kt levels
    "kt_male_sims": ens_levels_male[:, :, 0],
    "kt_female_sims": ens_levels_female[:, :, 0],
    "kt_male_mbc": ens_levels_male_mbc[:, :, 0],
    "kt_female_mbc": ens_levels_female_mbc[:, :, 0],
    # Metadata
    "forecast_years": forecast_years,
    "n_sims": N_SIMULATIONS,
    "n_years_ahead": N_YEARS_AHEAD,
    "seeds": SEEDS,
    "seed_rmses": seed_rmses,
    "bias_male": ens_bias_male,
    "bias_female": ens_bias_female,
    "process_std_male": sigma_res_male,
    "process_std_female": sigma_res_female,
    "countries": list(COUNTRIES.keys()),
    "method": "5_seed_ensemble_corrected_sigma",
}

with open(os.path.join(PROCESSED_DIR, "forecasting_assets_ensemble.pkl"), "wb") as f:
    pickle.dump(ensemble_assets, f)

df_comp.to_csv(os.path.join(PROCESSED_DIR, "ensemble_comparison.csv"), index=False)
df_scr.to_csv(os.path.join(PROCESSED_DIR, "scr_ensemble.csv"), index=False)

print("Ensemble assets saved:")
print(f"  {PROCESSED_DIR}forecasting_assets_ensemble.pkl")
print(f"  {PROCESSED_DIR}ensemble_comparison.csv")
print(f"  {PROCESSED_DIR}scr_ensemble.csv")
print(f"  Models: {MODELS_DIR}ainn_seed_*.keras (5 models)")
print(f"\nNotebook 08 complete.")
print(f"Step 1 + Step 2 of improvement plan are done.")
print(f"Next: evaluate final results and decide whether to proceed to Step 3 (teacher forcing) or move to paper drafting.")